In [ ]:
from __future__ import annotations
from pathlib import Path

import fsspec, pickle

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import manifold_dynamics.neural_utils as nu
import manifold_dynamics.paths as pth
import manifold_dynamics.spike_response_stats as srs
import manifold_dynamics.tuning_utils as tut
import visionlab_utils.storage as vst

In [ ]:
# ROI name, should be 3 parts (index.label.category)
target = "07.MF1.F"
target_parts = target.split(".")
roi_label = f"{int(target_parts[0]):02d}.{target_parts[1]}.{target_parts[2]}"

# load in multi-session ROI data, binned to PSTH
raster_4d = nu.significant_trial_raster(target, alpha=0.05, bin_size_ms=20)
raster_3d = np.nanmean(raster_4d, axis=3)

# get top-k value for ROI
topk_local = vst.fetch(f"{pth.OTHERS}/topk_vals.pkl")
with open(topk_local, "rb") as f:
    topk_vals = pickle.load(f)
top_k = int(topk_vals[roi_label]["k"])

In [ ]:
ONSET = 50
RESP = slice(ONSET + 50, ONSET + 220)
BASE = slice(ONSET - 50, ONSET + 0)

X = np.array(raster_3d)
resp = np.nanmean(X[:, RESP, :], axis=(0, 1))       # (images,)
base_mean = np.nanmean(X[:, BASE, :], axis=(0, 1))
base_std = np.nanstd(X[:, BASE, :], axis=(0, 1))

# avoid divide-by-zero
base_std = np.where(base_std == 0, np.nan, base_std)

zscores = (resp - base_mean) / base_std
scores = tut.rank_images_by_response(raster_3d)

In [ ]:
avg_resp = np.nanmean(zscores[scores][:top_k])
print(avg_resp)
sns.histplot(zscores)

In [ ]:
customp = sns.color_palette('coolwarm_r', len(scores))

fig, ax = plt.subplots(1, 1, figsize=(10, 4))
# vertical sticks from 0 to score
ax.vlines(range(len(scores)), 0, zscores[scores], 
          colors=customp, linewidth=0.6)

## ED z-score (relative to random)

In [ ]:
fs = fsspec.filesystem("s3")
files = fs.ls("visionlab-members/amarvi/manifold-dynamics/timextime/ed_main")

# full s3 URIs
uris = [f"s3://{f}" for f in files]
local_paths = [vst.fetch(uri) for uri in uris]
rows = []
for fpath in local_paths:
    fpath = Path(fpath)

    with open(fpath, "rb") as f:
        ed = pickle.load(f)
    df_roi = ed.copy()
    rows.append(df_roi)

df = pd.concat(rows, ignore_index=True)

def zscore_group(g):
    mu = g.loc[g.condition == "random", "ED"].mean()
    sigma = g.loc[g.condition == "random", "ED"].std()
    g["ED_z"] = (g["ED"] - mu) / sigma
    return g

df = df.groupby(["roi"], group_keys=False).apply(zscore_group)

def target_group(target):
    s = str(target)

    # "middle value" after splitting on "."
    parts = s.split(".")
    middle = parts[1] if len(parts) >= 3 else s

    if middle == "Unknown":
        return "Unknown"
    elif "F" in middle:
        return "F"
    elif "B" in middle:
        return "B"
    elif "O" in middle:
        return "O"
    else:
        return "Other"

df["target_group"] = df["target"].apply(target_group)
df = df[df["condition"] != "random"]
df

In [ ]:
current = pd.read_csv("~/Desktop/HVRD/workspace/manifold-dynamics/aux_controls/sampling_strength_summary.csv")
current

In [ ]:
out = df.drop(["bootstrap"], axis=1)
out.to_csv("~/Desktop/HVRD/workspace/manifold-dynamics/aux_controls/zscore_sampling_strength_summary.csv")